# Advanced Problems with Solutions: Sending Exceptions to Python Generators

This notebook develops advanced skill with:

- `generator.throw(...)`
- `generator.close()`
- `GeneratorExit`
- `StopIteration.value`
- exception propagation and translation
- `yield from` delegation
- generator state inspection
- exception-driven control signals
- resource-safe cleanup
- coroutine-style state machines

Every problem includes a complete solution and executable checks.

> **Best-practice note:** Exceptions can be used as control signals, but ordinary data should usually be sent as ordinary values. Use exception signals only when they clearly represent out-of-band control, cancellation, rollback, reset, or failure.

## Learning objectives

After completing the notebook, you should be able to:

1. Predict exactly where an injected exception is raised.
2. Distinguish recovery, propagation, return, and exception translation.
3. Explain why `close()` and `throw(GeneratorExit)` are similar but not identical.
4. Build generator-based state machines that remain safe under failure.
5. Verify generator lifecycle states using `inspect.getgeneratorstate`.
6. Design delegated generators whose cleanup and exception behavior are testable.

## Shared utilities

In [1]:
from __future__ import annotations

from contextlib import contextmanager
from dataclasses import dataclass
from inspect import getgeneratorlocals, getgeneratorstate
from typing import Any, Generator, Iterable

In [2]:
def prime(gen: Generator) -> Generator:
    """Advance a generator to its first suspension point and return it."""
    next(gen)
    return gen


def state(gen: Generator) -> str:
    """Return a compact generator state name."""
    return getgeneratorstate(gen)


def capture_exception(callable_obj, *args, **kwargs):
    """Run a callable and return (result, exception). Exactly one is non-None."""
    try:
        return callable_obj(*args, **kwargs), None
    except BaseException as exc:
        return None, exc


print("Utilities loaded.")

Utilities loaded.


## Core behavior recap

When `g.throw(exc)` is called, Python raises `exc` at the generator's current suspension point.

Possible outcomes:

- The generator catches the exception and **yields**: that yielded value becomes the return value of `throw`.
- The generator does not catch it: the exception propagates to the caller.
- The generator catches it and **returns**: the caller receives `StopIteration`.
- The generator catches it and raises a different exception: the new exception propagates.

# Problem 1 — Uncaught injection and lifecycle verification

Create a generator that records received values and always records cleanup in `finally`.

Inject a `KeyError` that the generator does not catch. Verify:

1. the `KeyError` reaches the caller;
2. cleanup runs exactly once;
3. the generator becomes `GEN_CLOSED`.

## Solution 1

In [3]:
def recording_sink(events: list[tuple[str, Any]]):
    try:
        while True:
            item = yield
            events.append(("value", item))
    finally:
        events.append(("cleanup", None))


events = []
g = prime(recording_sink(events))

g.send(10)
g.send(20)

result, exc = capture_exception(g.throw, KeyError("injected failure"))

assert result is None
assert isinstance(exc, KeyError)
assert exc.args == ("injected failure",)
assert events == [
    ("value", 10),
    ("value", 20),
    ("cleanup", None),
]
assert state(g) == "GEN_CLOSED"

print("events:", events)
print("propagated:", type(exc).__name__, exc)
print("state:", state(g))

events: [('value', 10), ('value', 20), ('cleanup', None)]
propagated: KeyError 'injected failure'
state: GEN_CLOSED


### Key point

An uncaught exception terminates the generator. The `finally` block still runs before the exception reaches the caller.

# Problem 2 — Recover, yield an acknowledgement, and continue

Build a generator that catches a custom recoverable exception and yields an acknowledgement.

Demonstrate that:

- `throw(...)` returns the value yielded by the handler;
- the generator remains suspended;
- after advancing out of the handler, normal `send(...)` operations continue.

## Solution 2

In [4]:
class RecoverableError(Exception):
    pass


def resilient_consumer(events: list[Any]):
    while True:
        try:
            item = yield "READY"
            events.append(("accepted", item))
        except RecoverableError as exc:
            events.append(("recovered", str(exc)))
            yield f"RECOVERED:{exc}"


events = []
g = resilient_consumer(events)

assert next(g) == "READY"
assert g.send("alpha") == "READY"

ack = g.throw(RecoverableError("retry"))
assert ack == "RECOVERED:retry"
assert state(g) == "GEN_SUSPENDED"

# The generator is suspended at the yield inside the except block.
# Advance once to return to the main receive point.
assert next(g) == "READY"
assert g.send("beta") == "READY"

assert events == [
    ("accepted", "alpha"),
    ("recovered", "retry"),
    ("accepted", "beta"),
]

g.close()
print("acknowledgement:", ack)
print("events:", events)

acknowledgement: RECOVERED:retry
events: [('accepted', 'alpha'), ('recovered', 'retry'), ('accepted', 'beta')]


### Best-practice observation

A `yield` inside an exception handler creates an additional suspension point. Document that protocol clearly so callers know whether they must call `next(...)` before sending the next data item.

# Problem 3 — Return a final result through `StopIteration.value`

Implement an accumulator that accepts integers with `send(...)`.

When a `Finalize` exception is injected, return the accumulated total. Extract that total from `StopIteration.value`.

## Solution 3

In [5]:
class Finalize(Exception):
    pass


def accumulating_coroutine():
    total = 0
    while True:
        try:
            value = yield total
        except Finalize:
            return total
        else:
            total += value


g = accumulating_coroutine()

assert next(g) == 0
assert g.send(4) == 4
assert g.send(6) == 10
assert g.send(-3) == 7

result, exc = capture_exception(g.throw, Finalize())

assert result is None
assert isinstance(exc, StopIteration)
assert exc.value == 7
assert state(g) == "GEN_CLOSED"

print("final total from StopIteration.value:", exc.value)

final total from StopIteration.value: 7


### Key point

A generator's `return expression` becomes `StopIteration(expression)`. This is especially important when a generator is used through `yield from`.

# Problem 4 — Translate exceptions and preserve causality

Create a generator that converts a low-level exception into a domain-specific exception using explicit exception chaining.

Verify:

- the new exception reaches the caller;
- `__cause__` references the original exception;
- the generator closes.

## Solution 4

In [6]:
class LowLevelFailure(Exception):
    pass


class DomainFailure(Exception):
    pass


def translating_generator():
    try:
        while True:
            yield "READY"
    except LowLevelFailure as exc:
        raise DomainFailure("operation could not be completed") from exc


g = translating_generator()
assert next(g) == "READY"

_, exc = capture_exception(g.throw, LowLevelFailure("socket reset"))

assert isinstance(exc, DomainFailure)
assert isinstance(exc.__cause__, LowLevelFailure)
assert str(exc.__cause__) == "socket reset"
assert state(g) == "GEN_CLOSED"

print("raised:", repr(exc))
print("caused by:", repr(exc.__cause__))

raised: DomainFailure('operation could not be completed')
caused by: LowLevelFailure('socket reset')


### Best practice

Use `raise NewError(...) from exc` when the lower-level failure is useful diagnostic context. Use `from None` only when suppressing that context is intentional.

# Problem 5 — Compare `close()` with `throw(GeneratorExit)`

Use identical generators to compare the two operations.

Confirm that:

- both execute `finally`;
- `close()` normally returns `None`;
- direct `throw(GeneratorExit)` exposes `GeneratorExit` to the caller when it is not handled.

## Solution 5

In [7]:
def traced_generator(log: list[str]):
    try:
        while True:
            yield "ACTIVE"
    finally:
        log.append("cleanup")


close_log = []
g1 = traced_generator(close_log)
assert next(g1) == "ACTIVE"

close_result = g1.close()

assert close_result is None
assert close_log == ["cleanup"]
assert state(g1) == "GEN_CLOSED"


throw_log = []
g2 = traced_generator(throw_log)
assert next(g2) == "ACTIVE"

_, thrown = capture_exception(g2.throw, GeneratorExit())

assert isinstance(thrown, GeneratorExit)
assert throw_log == ["cleanup"]
assert state(g2) == "GEN_CLOSED"

print("close() result:", close_result)
print("throw(GeneratorExit) propagated:", type(thrown).__name__)

close() result: None
throw(GeneratorExit) propagated: GeneratorExit


### Important detail

`GeneratorExit` inherits directly from `BaseException`, not from `Exception`. A broad `except Exception:` does not catch it.

# Problem 6 — Detect illegal yielding during close

Python forbids a generator from yielding a value while handling `GeneratorExit` triggered by `close()`.

Construct a deliberately incorrect generator and verify that `close()` raises:

```python
RuntimeError: generator ignored GeneratorExit
```

Then close it safely with a second `close()` call.

## Solution 6

In [8]:
def bad_cleanup_generator():
    try:
        yield "OPEN"
    except GeneratorExit:
        # This is intentionally incorrect.
        yield "ILLEGAL-YIELD"


g = bad_cleanup_generator()
assert next(g) == "OPEN"

_, exc = capture_exception(g.close)

assert isinstance(exc, RuntimeError)
assert "ignored GeneratorExit" in str(exc)
assert state(g) == "GEN_SUSPENDED"

# A second close injects GeneratorExit at the illegal yield point.
g.close()
assert state(g) == "GEN_CLOSED"

print(type(exc).__name__ + ":", exc)
print("final state:", state(g))

RuntimeError: generator ignored GeneratorExit
final state: GEN_CLOSED


### Best practice

During `GeneratorExit`, perform cleanup and then return. Do not yield. Usually, place cleanup in `finally` and avoid catching `GeneratorExit` unless you have a specific reason.

# Problem 7 — Transaction state machine with commit and rollback signals

Implement an in-memory transaction manager.

Protocol:

- `send(value)` adds a value to the pending transaction;
- `throw(Commit)` persists all pending values;
- `throw(Rollback)` discards all pending values;
- `close()` discards any uncommitted values.

Every operation should return a snapshot showing persisted and pending values.

## Solution 7

In [9]:
class Commit(Exception):
    pass


class Rollback(Exception):
    pass


def transaction_manager(persisted: list[Any]):
    pending: list[Any] = []

    def snapshot():
        return {
            "persisted": tuple(persisted),
            "pending": tuple(pending),
        }

    try:
        while True:
            try:
                value = yield snapshot()
                pending.append(value)
            except Commit:
                persisted.extend(pending)
                pending.clear()
            except Rollback:
                pending.clear()
    finally:
        # Never leave an uncommitted transaction alive.
        pending.clear()


db = []
tx = transaction_manager(db)

assert next(tx) == {"persisted": (), "pending": ()}
assert tx.send("A") == {"persisted": (), "pending": ("A",)}
assert tx.send("B") == {"persisted": (), "pending": ("A", "B")}

after_commit = tx.throw(Commit())
assert after_commit == {"persisted": ("A", "B"), "pending": ()}

assert tx.send("C") == {
    "persisted": ("A", "B"),
    "pending": ("C",),
}

after_rollback = tx.throw(Rollback())
assert after_rollback == {
    "persisted": ("A", "B"),
    "pending": (),
}

assert tx.send("D") == {
    "persisted": ("A", "B"),
    "pending": ("D",),
}

tx.close()

assert db == ["A", "B"]
assert state(tx) == "GEN_CLOSED"

print("after commit:", after_commit)
print("after rollback:", after_rollback)
print("persisted after close:", db)

after commit: {'persisted': ('A', 'B'), 'pending': ()}
after rollback: {'persisted': ('A', 'B'), 'pending': ()}
persisted after close: ['A', 'B']


### Design note

Using custom exception classes gives control signals a separate channel from normal data. This is useful for transaction boundaries, cancellation, rollback, reset, and shutdown.

# Problem 8 — `yield from` delegation and exception routing

Create a child accumulator and a parent that delegates using `yield from`.

Requirements:

- values sent to the parent reach the child;
- `ResetChild` is handled inside the child;
- `FinishChild` makes the child return its total;
- the parent's final `StopIteration.value` is the child's returned total;
- cleanup order is observable.

## Solution 8

In [10]:
class ResetChild(Exception):
    pass


class FinishChild(Exception):
    pass


def child_accumulator(log: list[Any]):
    total = 0
    try:
        while True:
            try:
                value = yield total
            except ResetChild:
                total = 0
            except FinishChild:
                return total
            else:
                total += value
    finally:
        log.append("child-cleanup")


def delegating_parent(log: list[Any]):
    try:
        result = yield from child_accumulator(log)
        log.append(("child-result", result))
        return result
    finally:
        log.append("parent-cleanup")


log = []
g = delegating_parent(log)

assert next(g) == 0
assert g.send(5) == 5
assert g.send(7) == 12

# throw() is automatically forwarded through yield from.
assert g.throw(ResetChild()) == 0
assert g.send(3) == 3

_, exc = capture_exception(g.throw, FinishChild())

assert isinstance(exc, StopIteration)
assert exc.value == 3
assert log == [
    "child-cleanup",
    ("child-result", 3),
    "parent-cleanup",
]
assert state(g) == "GEN_CLOSED"

print("delegated result:", exc.value)
print("cleanup log:", log)

delegated result: 3
cleanup log: ['child-cleanup', ('child-result', 3), 'parent-cleanup']


### Key point

`yield from` forwards `send`, `throw`, and `close` to the delegated iterator when supported. It also captures the child's `StopIteration.value`.

# Problem 9 — Broadcast control exceptions to multiple workers

Build accumulator workers and a supervisor.

Each worker must:

- accept numeric values;
- respond to `Flush` by yielding `(worker_name, total, count)`;
- reset after the flush acknowledgement is consumed;
- return its current count when `CancelWorker` is injected.

The supervisor must safely flush and cancel every live worker.

## Solution 9

In [11]:
class Flush(Exception):
    pass


class CancelWorker(Exception):
    pass


def accumulator_worker(name: str):
    total = 0.0
    count = 0

    while True:
        try:
            value = yield
        except Flush:
            yield (name, total, count)
            total = 0.0
            count = 0
        except CancelWorker:
            return count
        else:
            total += float(value)
            count += 1


def make_worker(name: str):
    return prime(accumulator_worker(name))


def flush_worker(worker: Generator):
    report = worker.throw(Flush())
    # Leave the handler's acknowledgement yield and return to the receive point.
    next(worker)
    return report


def cancel_worker(worker: Generator):
    _, exc = capture_exception(worker.throw, CancelWorker())
    if not isinstance(exc, StopIteration):
        raise AssertionError(f"unexpected cancellation result: {exc!r}")
    return exc.value


workers = [make_worker("north"), make_worker("south")]

workers[0].send(10)
workers[0].send(20)
workers[1].send(5)

reports = [flush_worker(worker) for worker in workers]
assert reports == [
    ("north", 30.0, 2),
    ("south", 5.0, 1),
]

workers[0].send(7)
workers[1].send(11)
workers[1].send(13)

cancel_counts = [cancel_worker(worker) for worker in workers]
assert cancel_counts == [1, 2]
assert all(state(worker) == "GEN_CLOSED" for worker in workers)

print("flush reports:", reports)
print("counts at cancellation:", cancel_counts)

flush reports: [('north', 30.0, 2), ('south', 5.0, 1)]
counts at cancellation: [1, 2]


### Best-practice observation

Supervisors should centralize protocol details such as re-advancing after an acknowledgement yield. This keeps callers from accidentally sending data into the wrong suspension point.

# Problem 10 — Narrow exception handling with a signal hierarchy

Create a control-signal hierarchy:

- `ControlSignal`
- `SoftReset`
- `Pause`
- `Terminate`

Build a generator that catches only `ControlSignal` subclasses.

Verify that an unrelated `ValueError` is not swallowed and closes the generator.

## Solution 10

In [12]:
class ControlSignal(Exception):
    pass


class SoftReset(ControlSignal):
    pass


class Pause(ControlSignal):
    pass


class Terminate(ControlSignal):
    pass


def controlled_counter():
    count = 0
    paused = False

    while True:
        try:
            value = yield {"count": count, "paused": paused}
            if not paused:
                count += int(value)
        except SoftReset:
            count = 0
        except Pause:
            paused = not paused
        except Terminate:
            return {"count": count, "paused": paused}


g = controlled_counter()

assert next(g) == {"count": 0, "paused": False}
assert g.send(5) == {"count": 5, "paused": False}
assert g.throw(Pause()) == {"count": 5, "paused": True}
assert g.send(100) == {"count": 5, "paused": True}
assert g.throw(Pause()) == {"count": 5, "paused": False}
assert g.throw(SoftReset()) == {"count": 0, "paused": False}

_, exc = capture_exception(g.throw, ValueError("not a control signal"))

assert isinstance(exc, ValueError)
assert state(g) == "GEN_CLOSED"

print("unrelated exception propagated:", repr(exc))

unrelated exception propagated: ValueError('not a control signal')


### Best practice

Catch the narrowest exception types possible. Broad catches can accidentally absorb programming errors, cancellation, or process-level exceptions.

# Problem 11 — Understand how `contextmanager` injects exceptions

The `contextlib.contextmanager` decorator uses a generator protocol internally.

Create a context manager that:

- records opening and closing;
- catches and suppresses `ValueError`;
- does not suppress `TypeError`.

Test both paths.

## Solution 11

In [13]:
@contextmanager
def audit_scope(log: list[str]):
    log.append("open")
    try:
        yield
    except ValueError as exc:
        log.append(f"handled:{exc}")
        # Not re-raising suppresses this exception.
    finally:
        log.append("close")


log = []
with audit_scope(log):
    raise ValueError("recoverable")

assert log == [
    "open",
    "handled:recoverable",
    "close",
]


log2 = []
_, exc = capture_exception(
    lambda: (
        # A helper lambda cannot contain a with statement, so call a nested function.
        None
    )
)

def run_type_error_case():
    with audit_scope(log2):
        raise TypeError("not handled")

_, exc = capture_exception(run_type_error_case)

assert isinstance(exc, TypeError)
assert log2 == ["open", "close"]

print("suppressed ValueError log:", log)
print("propagated TypeError:", repr(exc))
print("TypeError log:", log2)

suppressed ValueError log: ['open', 'handled:recoverable', 'close']
propagated TypeError: TypeError('not handled')
TypeError log: ['open', 'close']


### Interpretation

When an exception leaves a `with` block, the context-manager machinery throws that exception into the generator at the `yield` point. Suppression occurs when the generator handles the exception and finishes normally.

# Problem 12 — Circuit breaker controlled by injected failures

Implement a circuit breaker coroutine.

Rules:

- normal value `"success"` resets failures and closes the circuit;
- `throw(ServiceFailure(...))` increments the failure count;
- at the threshold, state becomes `"OPEN"`;
- `throw(ResetBreaker())` resets the breaker;
- `throw(StopBreaker())` returns the final snapshot.

## Solution 12

In [14]:
class ServiceFailure(Exception):
    pass


class ResetBreaker(Exception):
    pass


class StopBreaker(Exception):
    pass


def circuit_breaker(threshold: int):
    if threshold <= 0:
        raise ValueError("threshold must be positive")

    failures = 0
    breaker_state = "CLOSED"
    last_error = None

    def snapshot():
        return {
            "state": breaker_state,
            "failures": failures,
            "last_error": last_error,
        }

    while True:
        try:
            event = yield snapshot()
            if event == "success":
                failures = 0
                breaker_state = "CLOSED"
                last_error = None
        except ServiceFailure as exc:
            failures += 1
            last_error = str(exc)
            if failures >= threshold:
                breaker_state = "OPEN"
        except ResetBreaker:
            failures = 0
            breaker_state = "CLOSED"
            last_error = None
        except StopBreaker:
            return snapshot()


g = circuit_breaker(threshold=3)

assert next(g)["state"] == "CLOSED"
assert g.throw(ServiceFailure("timeout-1"))["failures"] == 1
assert g.throw(ServiceFailure("timeout-2"))["state"] == "CLOSED"

opened = g.throw(ServiceFailure("timeout-3"))
assert opened == {
    "state": "OPEN",
    "failures": 3,
    "last_error": "timeout-3",
}

reset = g.throw(ResetBreaker())
assert reset == {
    "state": "CLOSED",
    "failures": 0,
    "last_error": None,
}

_, exc = capture_exception(g.throw, StopBreaker())

assert isinstance(exc, StopIteration)
assert exc.value == reset

print("opened snapshot:", opened)
print("final snapshot:", exc.value)

opened snapshot: {'state': 'OPEN', 'failures': 3, 'last_error': 'timeout-3'}
final snapshot: {'state': 'CLOSED', 'failures': 0, 'last_error': None}


# Problem 13 — Cleanup order across nested delegation

Build three generator layers using `yield from`.

Verify the cleanup order for:

1. `close()` on the outer generator;
2. an uncaught injected `RuntimeError`.

The deepest generator should clean up first.

## Solution 13

In [15]:
def layer_c(log: list[str]):
    try:
        while True:
            yield "C"
    finally:
        log.append("C-cleanup")


def layer_b(log: list[str]):
    try:
        yield from layer_c(log)
    finally:
        log.append("B-cleanup")


def layer_a(log: list[str]):
    try:
        yield from layer_b(log)
    finally:
        log.append("A-cleanup")


close_log = []
g1 = layer_a(close_log)
assert next(g1) == "C"
g1.close()

assert close_log == [
    "C-cleanup",
    "B-cleanup",
    "A-cleanup",
]


throw_log = []
g2 = layer_a(throw_log)
assert next(g2) == "C"

_, exc = capture_exception(g2.throw, RuntimeError("cascade"))

assert isinstance(exc, RuntimeError)
assert throw_log == [
    "C-cleanup",
    "B-cleanup",
    "A-cleanup",
]
assert state(g2) == "GEN_CLOSED"

print("close cleanup order:", close_log)
print("throw cleanup order:", throw_log)

close cleanup order: ['C-cleanup', 'B-cleanup', 'A-cleanup']
throw cleanup order: ['C-cleanup', 'B-cleanup', 'A-cleanup']


### Key point

Delegated cleanup unwinds from the innermost active generator outward, matching ordinary stack unwinding.

# Problem 14 — Capstone: exception-driven batch processor

Build a production-style batch processor with these signals:

- `CommitBatch`
- `RollbackBatch`
- `PauseProcessor`
- `ResumeProcessor`
- `ShutdownProcessor`

Requirements:

- normal values are appended to `pending` unless paused;
- commit moves pending values to `persisted`;
- rollback discards pending values;
- pause rejects normal data without corrupting state;
- shutdown returns a final snapshot through `StopIteration.value`;
- `finally` clears any remaining uncommitted data.

## Solution 14

In [16]:
class CommitBatch(Exception):
    pass


class RollbackBatch(Exception):
    pass


class PauseProcessor(Exception):
    pass


class ResumeProcessor(Exception):
    pass


class ShutdownProcessor(Exception):
    pass


@dataclass(frozen=True)
class BatchSnapshot:
    persisted: tuple[Any, ...]
    pending: tuple[Any, ...]
    paused: bool
    commits: int
    rollbacks: int


def batch_processor(persisted: list[Any]):
    pending: list[Any] = []
    paused = False
    commits = 0
    rollbacks = 0

    def snapshot():
        return BatchSnapshot(
            persisted=tuple(persisted),
            pending=tuple(pending),
            paused=paused,
            commits=commits,
            rollbacks=rollbacks,
        )

    try:
        while True:
            try:
                value = yield snapshot()
                if paused:
                    raise RuntimeError("processor is paused")
                pending.append(value)

            except CommitBatch:
                persisted.extend(pending)
                pending.clear()
                commits += 1

            except RollbackBatch:
                pending.clear()
                rollbacks += 1

            except PauseProcessor:
                paused = True

            except ResumeProcessor:
                paused = False

            except ShutdownProcessor:
                return snapshot()
    finally:
        pending.clear()


storage = []
processor = batch_processor(storage)

assert next(processor) == BatchSnapshot((), (), False, 0, 0)

assert processor.send("evt-1").pending == ("evt-1",)
assert processor.send("evt-2").pending == ("evt-1", "evt-2")

committed = processor.throw(CommitBatch())
assert committed == BatchSnapshot(
    persisted=("evt-1", "evt-2"),
    pending=(),
    paused=False,
    commits=1,
    rollbacks=0,
)

paused = processor.throw(PauseProcessor())
assert paused.paused is True

# A normal send while paused raises inside the generator and closes it,
# so a production protocol should avoid data sends while paused.
# We use a fresh processor below to demonstrate a safer caller-side guard.
processor.close()


storage2 = []
processor2 = batch_processor(storage2)
current = next(processor2)

def safe_send(proc: Generator, snapshot: BatchSnapshot, value: Any):
    if snapshot.paused:
        raise RuntimeError("caller refused to send while processor is paused")
    return proc.send(value)


current = safe_send(processor2, current, "x")
current = processor2.throw(PauseProcessor())

_, guard_error = capture_exception(safe_send, processor2, current, "blocked")
assert isinstance(guard_error, RuntimeError)
assert state(processor2) == "GEN_SUSPENDED"

current = processor2.throw(ResumeProcessor())
current = safe_send(processor2, current, "y")
current = processor2.throw(RollbackBatch())

assert current == BatchSnapshot(
    persisted=(),
    pending=(),
    paused=False,
    commits=0,
    rollbacks=1,
)

current = safe_send(processor2, current, "z")
current = processor2.throw(CommitBatch())

_, shutdown = capture_exception(processor2.throw, ShutdownProcessor())

assert isinstance(shutdown, StopIteration)
assert shutdown.value == BatchSnapshot(
    persisted=("z",),
    pending=(),
    paused=False,
    commits=1,
    rollbacks=1,
)
assert storage2 == ["z"]
assert state(processor2) == "GEN_CLOSED"

print("caller-side guard:", guard_error)
print("final snapshot:", shutdown.value)

caller-side guard: caller refused to send while processor is paused
final snapshot: BatchSnapshot(persisted=('z',), pending=(), paused=False, commits=1, rollbacks=1)


### Capstone review

This design separates:

- **data plane:** values sent with `send(...)`;
- **control plane:** commit, rollback, pause, resume, and shutdown signals injected with `throw(...)`;
- **lifecycle plane:** cleanup guaranteed by `finally`.

The caller-side `safe_send` guard is important. A protocol should prevent invalid transitions before they accidentally terminate the generator.

# Problem 15 — Inspect suspended local state without mutating it

Use `inspect.getgeneratorlocals` to inspect a live generator.

Verify that:

- pending transactional data is visible while suspended;
- a rollback changes the suspended local state;
- inspection does not advance the generator.

## Solution 15

In [17]:
storage = []
g = transaction_manager(storage)

next(g)
g.send("item-1")
g.send("item-2")

before_state = state(g)
before_locals = getgeneratorlocals(g)

assert before_state == "GEN_SUSPENDED"
assert before_locals["pending"] == ["item-1", "item-2"]

g.throw(Rollback())

after_state = state(g)
after_locals = getgeneratorlocals(g)

assert after_state == "GEN_SUSPENDED"
assert after_locals["pending"] == []
assert storage == []

g.close()

print("before rollback pending:", before_locals["pending"])
print("after rollback pending:", after_locals["pending"])

before rollback pending: []
after rollback pending: []


### Best practice

Generator introspection is useful for debugging and tests, but application logic should not normally depend on private suspended locals.

# Problem 16 — Design a safe reusable driver API

Raw generator protocols can be easy to misuse. Wrap the transaction generator in a small class that exposes explicit methods:

- `write(value)`
- `commit()`
- `rollback()`
- `close()`
- `snapshot`

The wrapper should hide priming and all direct `throw(...)` calls.

## Solution 16

In [18]:
class TransactionDriver:
    def __init__(self, persisted: list[Any]):
        self._gen = transaction_manager(persisted)
        self._snapshot = next(self._gen)
        self._closed = False

    @property
    def snapshot(self):
        return self._snapshot

    def _ensure_open(self):
        if self._closed:
            raise RuntimeError("transaction driver is closed")

    def write(self, value: Any):
        self._ensure_open()
        self._snapshot = self._gen.send(value)
        return self._snapshot

    def commit(self):
        self._ensure_open()
        self._snapshot = self._gen.throw(Commit())
        return self._snapshot

    def rollback(self):
        self._ensure_open()
        self._snapshot = self._gen.throw(Rollback())
        return self._snapshot

    def close(self):
        if not self._closed:
            self._gen.close()
            self._closed = True


persisted = []
driver = TransactionDriver(persisted)

driver.write(1)
driver.write(2)
assert driver.snapshot["pending"] == (1, 2)

driver.commit()
assert driver.snapshot == {
    "persisted": (1, 2),
    "pending": (),
}

driver.write(3)
driver.rollback()
assert driver.snapshot["pending"] == ()

driver.close()

_, exc = capture_exception(driver.write, 99)
assert isinstance(exc, RuntimeError)
assert persisted == [1, 2]

print("persisted:", persisted)
print("closed-driver protection:", exc)

persisted: [1, 2]
closed-driver protection: transaction driver is closed


## Final checklist

Use these practices when designing generator exception protocols:

1. Prime generators in one well-defined place.
2. Use custom exception classes for control signals.
3. Catch only the signals you intend to handle.
4. Keep cleanup in `finally`.
5. Never yield while closing after `GeneratorExit`.
6. Document every suspension point.
7. Use wrappers to hide fragile `send`/`throw` choreography.
8. Test `GEN_SUSPENDED` and `GEN_CLOSED` states.
9. Test both handled and unhandled exception paths.
10. Prefer ordinary values for ordinary data.

## Additional challenge prompts

These are intentionally left unsolved for further practice:

1. Add nested savepoints to the transaction manager.
2. Add a timeout signal that rolls back only the current batch.
3. Extend the circuit breaker with `"HALF_OPEN"` state.
4. Build a supervisor that restarts only workers that fail with recoverable exceptions.
5. Implement a delegated parser pipeline where a child can return statistics to its parent.
6. Add a context-manager wrapper to `TransactionDriver`.
7. Create property-based tests for arbitrary sequences of send/commit/rollback operations.
8. Compare this generator protocol with an equivalent `asyncio` implementation.